# 데이터 재구조화 소개 및 데이터 불러오기 

## 데이터 가로 결합 

```python
width_key_number = 2687

width_sample_train = train[train['Auction_key'] == width_key_number][['Auction_key', 'road_name', 'Hammer_price']].reset_index(drop=True) # 인덱스를 초기화 
display(width_sample_train)
# 경매 결과 데이터 중 가장 최신의 상태 가져오기
max_seq_value = result[result['Auction_key'] == width_key_number]['Auction_seq'].max()
width_sample_result = result[(result['Auction_key'] == width_key_number) & (result['Auction_seq'] == max_seq_value)][['Auction_date', 'Auction_results']].reset_index(drop=True)

display(width_sample_train)
display(width_sample_result)
```

===

```

Auction_key	road_name	Hammer_price
0	2687	해운대해변로	760000000
Auction_key	road_name	Hammer_price
0	2687	해운대해변로	760000000
Auction_date	Auction_results
0	2018-06-14 00:00:00	배당

```

```python
width_concat = pd.concat([width_sample_train, width_sample_result], axis = 1)
width_concat
```

===

```
Auction_key	road_name	Hammer_price	Auction_date	Auction_results
0	2687	해운대해변로	760000000	2018-06-14 00:00:00	배당


```



## reset_index 함수 설명

`reset_index`는 판다스(Pandas)에서 데이터프레임의 엉클어진 인덱스(행 번호)를 기본값(0, 1, 2...)으로 초기화해 주는 역할을 합니다.
데이터를 필터링하거나 정렬, 그룹화(`groupby`) 연산을 하고 나면 인덱스가 순서대로 되어 있지 않거나 특정 컬럼이 인덱스로 빠지는 경우가 생깁니다. 이때 `reset_index()`를 사용하면 인덱스를 처음처럼 0, 1, 2... 순서로 깔끔하게 되돌릴 수 있습니다.

### 주요 파라미터(옵션)
- **`drop=True`**: 기존 인덱스를 새로운 컬럼으로 빼지 않고 완전히 버립니다. (가장 자주 쓰임)
- **`inplace=True`**: 변경된 사항을 새로운 변수에 담지 않고, 원본 데이터프레임에 바로 덮어씌웁니다.

### 추가 자료
- [Pandas 공식 문서 (reset_index)](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reset_index.html)

In [1]:
import pandas as pd

# 임의의 데이터프레임 생성
data = {'이름': ['철수', '영희', '민수'], '나이': [25, 22, 30]}
df = pd.DataFrame(data)

# 인덱스를 엉클어지게 만들기 위해 정렬 수행
df_sorted = df.sort_values(by='나이')
print("--- 정렬 후 (인덱스가 섞임) ---")
print(df_sorted)
# 출력:
#    이름  나이
# 1  영희  22
# 0  철수  25
# 2  민수  30

# 1. 기본 reset_index() 사용 (기존 인덱스가 'index'라는 새로운 열로 들어감)
df_reset = df_sorted.reset_index()
print("\n--- 기본 reset_index() 적용 ---")
print(df_reset)

# 2. drop=True 옵션 사용 (기존 인덱스 깔끔하게 버리기)
df_clean = df_sorted.reset_index(drop=True)
print("\n--- drop=True 옵션 적용 ---")
print(df_clean)
# 출력:
#    이름  나이
# 0  영희  22
# 1  철수  25
# 2  민수  30

--- 정렬 후 (인덱스가 섞임) ---
   이름  나이
1  영희  22
0  철수  25
2  민수  30

--- 기본 reset_index() 적용 ---
   index  이름  나이
0      1  영희  22
1      0  철수  25
2      2  민수  30

--- drop=True 옵션 적용 ---
   이름  나이
0  영희  22
1  철수  25
2  민수  30


# 데이터 세로 결합 

- 여러 기간, 지역, 조건 등에 따른 데이터를 하나로 합쳐 전체적인 분석을 하고자 할 때 사용, 
특정 조건을 만족하는 데이터만을 모아서 별도의 분석을 수행하고자 할 때 사용 

[문제 2]
아래 지시사항을 바탕으로 빈칸을 채우고, result 데이터프레임의 각 Auction_key 별 마지막 값을 가진 행을 추출한 데이터프레임 last_auction_df를 만들어보세요.

unique 메서드를 사용하여 Auction_key의 고유한 값을 추출하고, 그 고유값 데이터만을 선택하여 auction_data 변수에 저장합니다.
max 집계함수를 사용해서 Auction_seq 컬럼에서 가장 큰 값을 가져옵니다.
판다스의 함수를 사용하여 두 데이터프레임을 세로로 결합합니다.

```python
from tqdm import tqdm

last_auction_df = pd.DataFrame()

for auction_key in tqdm(result['Auction_key'].unique()):
    auction_data = result[result['Auction_key'] == auction_key]
    
    last_auction = auction_data[auction_data['Auction_seq'] == auction_data['Auction_seq'].max()]
    
    last_auction_df = pd.concat([last_auction_df, last_auction], axis=0)

print(f"원래 result 데이터프레임의 길이 :{len(result)}")
print(f"세로로 데이터를 이어붙인 데이터프레임의 길이: {len(last_auction_df)}")
last_auction_df.head(3)
```

## tqdm reference 

- [tqdm library](./tip/tqdm.md#tqdm-라이브러리-설명)

## 멀티 인덱스 설정 및 활용

```python
multi_index_result = result.set_index(['Auction_key', 'Auction_seq'])
multi_index_result
```

--- 

## result 

```
		Auction_date	Appraisal_price	Minimum_sales_price	Auction_results
Auction_key	Auction_seq				
1	1	2011-06-21 00:00:00	313000000	313000000	변경
2	2011-12-13 00:00:00	313000000	313000000	변경
3	2016-05-17 00:00:00	298000000	298000000	유찰
4	2016-06-21 00:00:00	298000000	238400000	유찰
5	2016-07-26 00:00:00	298000000	190720000	유찰
...	...	...	...	...	...
2761	2	2018-03-28 00:00:00	315000000	252000000	낙찰
3	2018-05-31 00:00:00	315000000	252000000	배당
2762	1	2018-02-21 00:00:00	135000000	135000000	유찰
2	2018-03-28 00:00:00	135000000	108000000	낙찰
3	2018-05-30 00:00:00	135000000	108000000	배당
```


## join

- `df1.join(df2)` --> df1의 인덱스를 기준으로 df2와 병합한다. 

```python
set_index_train = need_join_train.set_index('Auction_key')
set_index_auction = need_join_last_auction.set_index('Auction_key')

complete_join = set_index_train.join(set_index_auction)

display(complete_join)
print(f"need_join_train 데이터 길이 : {len(need_join_train)}")
print(f"need_join_last_auction 데이터 길이 : {len(need_join_last_auction)}")
print(f"complete_join 데이터 길이 : {len(complete_join)}")
```

- index로 설정된 키를 기준으로 데이터를 병합하고자 할 때 쓰임 

# merge

- 두 데이터 프레임을 하나 이상의 공통 열을 기준으로 병합하는 메서드 
- join과 마찬가지로 공통된 값을 가진 속성을 선택하여 병합 할 수 있다. 
- 다른 출처의 데이터를 병합하여 다양한 정보를 얻거나, 데이터를 정리할 때 사용

```python
left_merge_complete = need_merge_train.merge(
            need_merge_last_auction, 
            on='Auction_key',
            how='left',
            suffixes=('_train', '_auction'),
            indicator=True
)      

display(left_merge_complete.head(2))
print(f"need_merge_train 데이터 길이 : {len(need_merge_train)}")
print(f"need_merge_last_auction 데이터 길이 : {len(need_merge_last_auction)}")
print(f"left_merge_complete 데이터 길이 : {len(left_merge_complete)}")
```